# Election Program Parser

#### This script is useful for election program pdf's

In [2]:
import os
import re
import csv
import fitz  # PyMuPDF
import pytesseract
import pandas as pd
from datetime import datetime
from PIL import Image
import PyPDF2
from rapidfuzz import process, fuzz
from IPython.display import display, HTML
import numpy as np

import re
import unicodedata
from pathlib import Path
from typing import Optional, Dict, Any, Tuple

folder_path = r"C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\verkiezingen\Partijprogramma's volledig" #set location of file 
output_csv = "election_programs_parsed_2025.csv" #set name of output file

In [3]:
# Check if folder is accessible
print("Exists:", os.path.exists(folder_path))
print("Is Directory:", os.path.isdir(folder_path))

# List first few files
try:
    files = os.listdir(folder_path)
    print(f"Total files detected: {len(files)}")
    print("First 10 files:", files[:10])
except Exception as e:
    print(f"Error listing files: {e}")

pdf_files = [f for f in os.listdir(folder_path) if f.endswith(('.pdf', '.PDF'))]

print(f"Total PDFs found after fix: {len(pdf_files)}")
print("First 10 PDFs:", pdf_files[:10])

Exists: True
Is Directory: True
Total files detected: 216
First 10 files: ['2025_BIJ1_Tweede%20Kamer%20Verkiezingsprogramma%202025%20-%20Doe%20eerlijk.pdf', '50PLUS-PartijArchief-30-09-2025.pdf', '50PLUS_2019-verkiezingsprogramma-EP-20180917.pdf', 'ALDE%20Manifesto%202019.pdf', 'AltenaLokaal_Altena_GR2018.pdf', 'Basisinkomen%20Partij%202017%20Verkiezingsprogramma%20in%20het%20kort.pdf', 'Basisinkomenpartij%20verkiezingsprogramma%20TK%202021.pdf', 'BBB%20Concept%20Verkiezingsprogramma%20TK%202023.pdf', 'BBB%20Verkiezingsprogramma%20EP%202024%20Samenvatting.pdf', 'BBB%20Verkiezingsprogramma%20EP%202024.pdf']
Total PDFs found after fix: 216
First 10 PDFs: ['2025_BIJ1_Tweede%20Kamer%20Verkiezingsprogramma%202025%20-%20Doe%20eerlijk.pdf', '50PLUS-PartijArchief-30-09-2025.pdf', '50PLUS_2019-verkiezingsprogramma-EP-20180917.pdf', 'ALDE%20Manifesto%202019.pdf', 'AltenaLokaal_Altena_GR2018.pdf', 'Basisinkomen%20Partij%202017%20Verkiezingsprogramma%20in%20het%20kort.pdf', 'Basisinkomenpartij%20v

In [4]:
def extract_text_from_pdf(pdf_path, max_ocr_pages=1):
    """Extract text using PyMuPDF; if a page fails, OCR that page."""
    text_parts = []
    with fitz.open(pdf_path) as doc:
        for page_num in range(len(doc)):
            try:
                t = doc[page_num].get_text("text")  # block-aware plain text
            except Exception:
                t = ""
            if not t.strip():
                # Fallback OCR (only first N pages to keep it fast)
                if page_num < max_ocr_pages:
                    try:
                        page = doc.load_page(page_num)
                        pix = page.get_pixmap()
                        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                        import pytesseract
                        t = pytesseract.image_to_string(img, lang="nld+eng")
                    except Exception:
                        t = ""
            text_parts.append(t)
    return "\n".join(text_parts)

In [5]:
test_pdf = os.path.join(folder_path, pdf_files[0])  # Pick first PDF
print(f"Testing file: {test_pdf}")

try:
    text = extract_text_from_pdf(test_pdf)  # Run extraction
    print("Extracted text (first 500 characters):")
    print(text[:500])  # Show first 500 characters of extracted text
except Exception as e:
    print(f"Error extracting text from {test_pdf}: {e}")

Testing file: C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\verkiezingen\Partijprogramma's volledig\2025_BIJ1_Tweede%20Kamer%20Verkiezingsprogramma%202025%20-%20Doe%20eerlijk.pdf
Extracted text (first 500 characters):
Doe eerlijk.
Doe eerlijk.
STEM
Tweede Kamer 
verkiezings- 
programma

Tweede Kamer Verkiezingsprogramma BIJ1 2025 - Doe eerlijk. 

Kameraden,
Er zijn dingen waar we allemaal om geven. Een 
warm thuis waar we veilig zijn. Werk waarmee we 
fatsoenlijk kunnen leven. Zorg als we ziek zijn. 
Onderwijs dat onze kinderen kansen geeft. De 
zekerheid dat we de huur kunnen betalen en nog 
geld overhouden voor een leven dat de moeite 
waard is. Dit zijn geen radicale eisen, dit zijn 
menselijke basisbehoef


In [6]:
import re
import unicodedata
from typing import Optional, Tuple
from urllib.parse import unquote

# 1) Normalisatie
def norm(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.lower()
    s = s.replace("+", " plus ")
    s = re.sub(r"[_\-/.]", " ", s)        # separators -> spaties
    s = s.replace("’", "'").replace("'", "")
    s = s.replace(".", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 2) Aliassen
PARTY_ALIASES = {
    "VVD": ["vvd", "volkspartij voor vrijheid en democratie"],
    "PvdA": ["pvda", "partij van de arbeid", "p v d a"],
    "GroenLinks": ["groenlinks", "groen links", "gl"],
    "D66": ["d66", "democraten66", "democraten 66"],
    "CDA": ["cda", "christen democratisch appèl", "christen democratisch appel"],
    "ChristenUnie": ["christenunie", "christen unie", "cu"],
    "SGP": ["sgp"],
    "PvdD": ["pvdd", "partij voor de dieren", "p v d d"],
    "PVV": ["pvv", "partij voor de vrijheid"],
    "FvD": ["fvd", "forum voor democratie", "forum v democratie"],
    "50PLUS": ["50plus", "50 plus"],
    "DENK": ["denk"],
    "JA21": ["ja21", "ja 21"],
    "BBB": ["bbb", "boer burger beweging", "boerburgerbeweging"],
    "BIJ1": ["bij1", "bij 1"],
    "Volt": ["volt nederland", "volt"],
    "BVNL": ["bvnl", "belang van nederland"],
    "Piratenpartij": ["piratenpartij"],
    "NIDA": ["nida"],
    "Code Oranje": ["code oranje", "codeoranje"],
    "Splinter": ["splinter"],
}

# 3) Vooraf genormaliseerde varianten (DIT maakt CANON_TO_NORMS aan)
CANON_TO_NORMS = {
    canon: list({norm(v) for v in ([canon] + aliases)})
    for canon, aliases in PARTY_ALIASES.items()
}
ALL_VARIANTS = [(canon, v) for canon, vs in CANON_TO_NORMS.items() for v in vs]
ALL_VARIANT_STRINGS = [v for _, v in ALL_VARIANTS]

# 4) Optionele fuzzy (RapidFuzz)
try:
    from rapidfuzz import process, fuzz
    HAS_RAPIDFUZZ = True
except Exception:
    HAS_RAPIDFUZZ = False

# 5) Jaarextractie helper (pakt YYYY, YYYYMMDD, DDMMYYYY, etc.)
def _extract_year_from_string(s: str):
    if not s:
        return None
    years = []

    # YYYYMMDD
    years += [int(y) for (y, _, _) in re.findall(r'(20\d{2})(0[1-9]|1[0-2])(0[1-9]|[12]\d|3[01])', s)]
    # DDMMYYYY
    years += [int(y) for (_, _, y, _) in re.findall(r'(0[1-9]|[12]\d|3[01])(0[1-9]|1[0-2])((19|20)\d{2})', s)]
    # YYYY-MM-DD
    years += [int(y) for (y, _, _) in re.findall(r'(20\d{2})[-/_](0[1-9]|1[0-2])[-/_](0[1-9]|[12]\d|3[01])', s)]
    # DD-MM-YYYY
    years += [int(y) for (_, _, y, _) in re.findall(r'(0[1-9]|[12]\d|3[01])[-/_](0[1-9]|1[0-2])((19|20)\d{2})', s)]

    # LOS JAAR: gebruik lookarounds -> werkt ook voor 'GR2018', 'EP2019', 'TK2021'
    years += [int(y) for y in re.findall(r'(?<!\d)((?:19|20)\d{2})(?!\d)', s)]

    years = [y for y in years if 1900 <= y <= 2099]
    return max(years) if years else None

# 6) Hoofdfunctie: partij + jaar
def match_party_and_year(
    text: Optional[str] = None,
    filename: Optional[str] = None,
    fuzzy: bool = True,
    max_chars: int = 120_000,
) -> Tuple[Optional[str], str, Optional[int], Optional[str], Optional[int]]:
    """
    Retourneert: (party, method, score, source, year)
      - party: canonical partijnaam of None
      - method: 'exact' | 'fuzzy' | 'none'
      - score: fuzzy score (int) of None
      - source: 'text' of 'filename' (waar de match vandaan kwam)
      - year: gevonden jaartal (int) of None
    """
    sources = []
    if text:
        sources.append(("text", norm(text[:max_chars])))
    if filename:
        sources.append(("filename", norm(filename)))

    # --- Partij: exact tellen ---
    best_party, best_count, best_source = None, 0, None
    for src_name, src in sources:
        counts = {canon: 0 for canon in CANON_TO_NORMS}
        for canon, variants in CANON_TO_NORMS.items():
            for v in variants:
                pattern = r"\b" + re.escape(v) + r"\b"
                hits = len(re.findall(pattern, src))
                if hits:
                    counts[canon] += hits
        if counts:
            canon_here, cnt_here = max(counts.items(), key=lambda kv: kv[1])
            if cnt_here > best_count:
                best_party, best_count, best_source = canon_here, cnt_here, src_name

    method, score = ("exact", None) if best_count > 0 else ("none", None)

    # --- Fuzzy fallback ---
    if best_count == 0 and fuzzy and HAS_RAPIDFUZZ:
        best = (None, -1, None, None)  # party, score, method, source
        for src_name, src in sources:
            hit = process.extractOne(src, ALL_VARIANT_STRINGS, scorer=fuzz.partial_ratio)
            if hit:
                variant_str, sc, _ = hit
                if sc > best[1]:
                    for canon, v in ALL_VARIANTS:
                        if v == variant_str:
                            best = (canon, int(sc), "fuzzy", src_name)
                            break
        if best[0] and best[1] >= 88:
            best_party, score, method, best_source = best[0], best[1], best[2], best[3]

    # --- Jaar: eerst filename (URL-decode), anders tekst ---
    year = None
    if filename:
        fname_raw = unquote(filename)
        year = _extract_year_from_string(fname_raw) or _extract_year_from_string(norm(fname_raw))
    if year is None and text:
        year = _extract_year_from_string(text)

    return best_party, method, score, best_source, year


In [7]:
import os
import re
import pandas as pd

def clean_text(text: str) -> str:
    """Clean unwanted characters, including literal '\n', soft hyphens, ligatures."""
    if not isinstance(text, str):
        return text

    # Replace common PDF artifacts
    text = (text
            .replace('\xa0', ' ')   # non-breaking space
            .replace('\r', ' ')
            .replace(',', ' ')
            .replace('\u00ad', '')  # soft hyphen
            .replace('\ufb01', 'fi')# ﬁ ligature
            .replace('\ufb02', 'fl')# ﬂ ligature
           )

    # Remove both literal and real newlines/tabs -> single space
    text = re.sub(r'(\\n|\\r|\\t|\n|\r|\t)+', ' ', text)

    # Collapse multiple spaces and tidy punctuation spacing
    text = re.sub(r'\s{2,}', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\s+([,.;:!?])', r'\1', text)

    return text.strip()


def find_date_in_text(text: str):
    """(Optioneel) Fallback: vind datum-string in vrije tekst."""
    date_patterns = [
        re.compile(r'\b(\d{1,2})\s+(januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+(\d{4})', re.I),
        re.compile(r'\b(\d{1,2})\s+(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d{4})', re.I),
        re.compile(r'\b(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b'),
        re.compile(r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b', re.I),
        re.compile(r'\b(19|20)\d{2}\b'),  # losse jaartallen als allerlaatste optie
    ]
    t = clean_text(text)
    for pat in date_patterns:
        m = pat.search(t)
        if m:
            return m.group(0)
    return None


def parse_pdf_text(text: str, filename: str):
    """Return (party, year, body) using full text + filename."""
    # Zorg dat match_party_and_year al bestaat en ook URL-decoding/jaar doet
    party, method, score, best_source, year = match_party_and_year(text=text, filename=filename)

    # Fallback: als year nog None is, probeer uit tekst een datumstring te pakken en daar jaar uit te trekken
    if year is None:
        ds = find_date_in_text(text)
        if ds:
            m = re.search(r'\b((?:19|20)\d{2})\b', ds)
            if m:
                year = int(m.group(1))

    body = clean_text(text)
    return party, year, body


def parse_pdfs_in_folder(folder_path: str):
    """Parse all PDFs in a folder and create a DataFrame."""
    data = []
    missing_years = []
    skip_files = set()  # voeg specifieke bestandsnamen toe indien nodig

    for filename in os.listdir(folder_path):
        if filename in skip_files or not filename.lower().endswith('.pdf'):
            continue
        path = os.path.join(folder_path, filename)
        raw_text = extract_text_from_pdf(path)  # jouw bestaande functie
        party, year, body = parse_pdf_text(raw_text, filename)
        data.append({'party': party, 'year': year, 'body': body})

        if year is None:
            missing_years.append(filename)

    if missing_years:
        print(f"PDFs missing years (first 10): {missing_years[:10]}{' ...' if len(missing_years) > 10 else ''}")
    return pd.DataFrame(data)

df = parse_pdfs_in_folder(folder_path)
df


,party,year,body
0,BIJ1,2025,Doe eerlijk. Doe eerlijk. STEM Tweede Kamer ve...
1,50PLUS,2025,Verkiezingsprogramma 2025 - 2029 1 Verkiezings...
2,50PLUS,2019,1 Onze toekomst in Europa. Solidair met jong e...
3,GroenLinks,2019,Resolution: Freedom opportunity prosperity: th...
4,DENK,2018,LIJST 1 SAMEN MAKEN WIJ ER WERK VAN Vooraf Eco...
...,...,...,...
211,DENK,2025,1 VRIJ VERBOND VERKIEZINGS- PROGRAMMA 2025 – 2...
212,VVD,2019,PROGRAMMACOMMISSIE Ruben Brekelmans Debbie van...
213,DENK,2017,Concept verkiezingsprogramma 2017-2021 ZEKER N...
214,DENK,2017,VVD verkiezingsprogramma 2017-2021 ZEKER NEDER...


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   party   214 non-null    object
 1   year    216 non-null    int64 
 2   body    216 non-null    object
dtypes: int64(1), object(2)
memory usage: 5.2+ KB


In [9]:
mask_unbalanced = df['body'].astype(str).str.count('"') % 2 == 1
print('Ongebalanceerde quotes:', mask_unbalanced.sum())

Ongebalanceerde quotes: 7


In [10]:
# Choose where to save it
output_folder = os.path.dirname(folder_path)
output_path = os.path.join(output_folder, output_csv)

df.to_csv(os.path.join(output_folder, output_csv))


# df.to_csv(
#     output_path.replace('.csv', '.tsv'),
#     sep='\t',
#     index=False,
#     encoding='utf-8-sig',
#     quoting=csv.QUOTE_ALL,
#     doublequote=True,
#     escapechar='\\'
# )

print(f"✅ CSV saved to: {output_path}")

#open from text/csv in excel!

✅ CSV saved to: C:\Users\joly-\OneDrive - Universiteit Utrecht\UU\Dataschool\HUMAN\Data\verkiezingen\election_programs_parsed_2025.csv
